In [2]:
library(readxl)
library(readr)
library(dplyr)
library(ggplot2)
library(tidyr)
library(ggrepel)
library(networkD3)
library(tibble)
library(ggalluvial)
library(RColorBrewer)
library(scales)
library(viridis)

Warning message:
“package ‘readxl’ was built under R version 4.3.3”

Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Warning message:
“package ‘ggplot2’ was built under R version 4.3.3”
Warning message:
“package ‘networkD3’ was built under R version 4.3.3”
Warning message:
“package ‘ggalluvial’ was built under R version 4.3.3”
Warning message:
“package ‘scales’ was built under R version 4.3.3”

Attaching package: ‘scales’


The following object is masked from ‘package:readr’:

    col_factor


Warning message:
“package ‘viridis’ was built under R version 4.3.3”
Loading required package: viridisLite


Attaching package: ‘viridis’


The following object is masked from ‘package:scales’:

    viridis_pal




In [3]:
hormone <- read.table("/vol/projects/CIIM/cohorts_old/500FG/Molecular_phenotype/500FG_log2_hormone_levels.txt", header = TRUE, sep = "", stringsAsFactors = FALSE)
head(hormone)
dim(hormone)
any(is.na(hormone))
sum(is.na(hormone))

,B_ADNC,B_CORC,B_DESC,B_OHP,B_PROG,B_TESC,B_VITD,B_TSH,B_T4VR
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
HV001,0.7990873,-2.3959287,-2.5563933,0.1890338,-3.473931,3.5728897,5.930737,1.3479491,4.080922
HV002,1.9818527,-1.3219281,-0.5145732,1.2509616,-1.943416,3.6322682,6.409391,0.7402800,4.123989
HV003,0.6507646,-0.2515388,0.3785116,-0.9434165,-2.643856,-1.1202942,6.554589,1.2270633,4.394075
HV004,0.6415460,-0.4540316,-1.7369656,-1.8365013,-3.184425,-1.0291463,6.569856,0.9414820,4.080922
HV005,2.9467309,-0.8889687,0.4854268,0.5945485,-2.000000,0.4541759,5.857981,1.3219281,4.176075
HV006,1.4114262,-1.5145732,-1.3219281,-0.4540316,-2.643856,-0.9714308,6.321928,-0.1739702,4.196395


[1] 486   9

[1] TRUE

[1] 23

In [8]:
basicPhenos = read.csv("/vol/projects/yzhang/500FG_aging/output/02_Olink/Age_group_basicPhenos.csv")
rownames(basicPhenos) = basicPhenos$ID_500fg

## Filter a feature matrix and phenotype table to shared sample IDs and align rows in the same order
prep_filter_match <- function(feature_mat, basicPhenos) {
  idx <- intersect(rownames(feature_mat), rownames(basicPhenos))
  feature_filter <- feature_mat[idx, , drop = FALSE]
  pheno_match <- basicPhenos[rownames(feature_filter), , drop = FALSE]
  list(feature_filter = feature_filter, basicPhenos_filter_match = pheno_match)
}
# usage
res <- prep_filter_match(hormone, basicPhenos)
hormone_filter <- res$feature_filter
basicPhenos_filter_match <- res$basicPhenos_filter_match
##one missing gender value
basicPhenos_filter_match$Gender <- trimws(basicPhenos_filter_match$Gender)
basicPhenos_filter_match$Gender[basicPhenos_filter_match$Gender == ""] <- NA

head(hormone_filter)
dim(hormone_filter)
head(basicPhenos_filter_match)
dim(basicPhenos_filter_match)


,B_ADNC,B_CORC,B_DESC,B_OHP,B_PROG,B_TESC,B_VITD,B_TSH,B_T4VR
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
HV001,0.7990873,-2.3959287,-2.5563933,0.1890338,-3.473931,3.5728897,5.930737,1.3479491,4.080922
HV002,1.9818527,-1.3219281,-0.5145732,1.2509616,-1.943416,3.6322682,6.409391,0.7402800,4.123989
HV003,0.6507646,-0.2515388,0.3785116,-0.9434165,-2.643856,-1.1202942,6.554589,1.2270633,4.394075
HV004,0.6415460,-0.4540316,-1.7369656,-1.8365013,-3.184425,-1.0291463,6.569856,0.9414820,4.080922
HV005,2.9467309,-0.8889687,0.4854268,0.5945485,-2.000000,0.4541759,5.857981,1.3219281,4.176075
HV006,1.4114262,-1.5145732,-1.3219281,-0.4540316,-2.643856,-0.9714308,6.321928,-0.1739702,4.196395


[1] 479   9

,X,ID_500fg,Gender,Age,Height,Weight,BMI,LivingSituation,Education,Ethnicity,FarmAnimal,Pets,Travel,DayInMonth,monthinYear,dayInYear,year,AgeGroup
,<int>,<chr>,<chr>,<int>,<int>,<int>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<chr>
HV001,304,HV001,male,27,178,70,22.09317,Married,University degree,European,Once a month,Yes,No,24,7,204,2013,Group 5
HV002,305,HV002,male,26,178,68,21.46194,Cohabitation,University degree,European,Rarely,No,Yes,24,7,204,2013,Group 4
HV003,257,HV003,female,24,173,68,22.72044,Married,University degree,European,Rarely,Yes,Yes,26,8,236,2013,Group 4
HV004,258,HV004,female,20,173,65,21.71807,Married,University degree,European,Rarely,Yes,No,26,8,236,2013,Group 2
HV005,254,HV005,female,22,178,64,20.19947,Married,University student,European,Once a month,Yes,Yes,5,9,245,2013,Group 3
HV006,530,HV006,female,24,168,58,20.54989,Married,University degree,European,Rarely,Yes,No,NA,NA,NA,NA,Group 4


[1] 479  18

In [9]:
####model 2: age + hormone using FDR age not scale + gender
all_results <- list()

for (i in seq_len(ncol(hormone_filter))) {

  # Build a minimal modeling table
  data <- data.frame(
    feature = hormone_filter[, i],
    age = basicPhenos_filter_match$Age,
    Gender = factor(basicPhenos_filter_match$Gender)  # convert chr to factor
  )

  # Linear regression
  mod <- lm(feature ~ age + Gender, data = data)

  # Extract coefficient and p-value for age (not Gender)
  cf <- summary(mod)$coefficients
  res <- cf["age", c("Estimate", "Pr(>|t|)")]

  # Store results
  result <- data.frame(
    feature  = colnames(hormone_filter)[i],
    estimate = res["Estimate"],
    p        = res["Pr(>|t|)"]
  )

  # Signed -log10(p)
  result$value <- -log10(result$p) * result$estimate

  all_results[[i]] <- result
}

final_result <- do.call(rbind, all_results)
final_result$padj <- p.adjust(final_result$p, method = "BH")
final_result$sig <- ifelse(final_result$padj < 0.05, "sig", "no")

head(final_result)
dim(final_result)
table(final_result$sig)
write.csv(final_result, file = "/vol/projects/yzhang/500FG_aging/output/06_hormone/hormone_age_liner_gender_FDR.csv")

,feature,estimate,p,value,padj,sig
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
Estimate,B_ADNC,-0.02501094,4.283096e-26,-0.63448350,3.854787e-25,sig
Estimate1,B_CORC,-0.02015736,2.037838e-17,-0.33644307,9.170270e-17,sig
Estimate2,B_DESC,-0.02048826,3.050586e-07,-0.13349364,6.663984e-07,sig
Estimate3,B_OHP,-0.01130764,3.791113e-03,-0.02737844,4.874289e-03,sig
Estimate4,B_PROG,-0.01576901,3.129267e-03,-0.03949439,4.693901e-03,sig
Estimate5,B_TESC,-0.01334730,2.096557e-09,-0.11583443,6.289670e-09,sig


[1] 9 6


 no sig 
  1   8 

In [36]:
# 添加列：显著性分类（上调/下调/非显著）
final_result <- final_result %>% 
  mutate(
    sig = ifelse(p < 0.05 & estimate > 0, "upregulated",
                 ifelse(p < 0.05 & estimate < 0, "downregulated", "not significant")),
    log_p = -log10(pmax(p, .Machine$double.eps))  # 避免 p 为零导致计算错误
  )

# 分别筛选上调和下调中 log_p 最大的前 4 个
top3_upregulated <- final_result %>%
  filter(sig == "upregulated") %>%
  arrange(desc(log_p)) %>%
   head(3)


top3_downregulated <- final_result %>%
  filter(sig == "downregulated") %>%
  arrange(desc(log_p)) %>%
  head(3)

# 合并上调和下调的前 3 个显著点
top4_combined <- bind_rows(top3_upregulated, top3_downregulated)

# 设置保存火山图的分辨率和尺寸
ppi <- 300
png("/vol/projects/yzhang/500FG_aging/output/06_hormone/hormone_age_top3.png", 
    width = 5 * ppi, height = 4 * ppi, res = ppi)

# 绘制火山图
ggplot(final_result, aes(x = estimate, y = log_p)) + 
  geom_point(aes(color = sig), size = 3, alpha = 0.6) +  # 显著性颜色
  scale_color_manual(values = c(
    "upregulated" = "#FFCC99", 
    "downregulated" = "#B3DE69", 
    "not significant" = "grey"
  )) + 
  theme_classic() + 
  xlab("Estimate (Effect Size)") +
  ylab("-log10(p)") +
  ggtitle("Hormone ~ Age  Volcano Plot") +
  # 显示显著性统计信息
  annotate(
    "text", 
    x = max(final_result$estimate, na.rm = TRUE) -0.1, 
    y = max(final_result$log_p, na.rm = TRUE) - 4.5, 
    label = paste0(
     "Upregulated: ", sum(final_result$sig == "upregulated"), 
     " (", round(mean(final_result$sig == "upregulated") * 100, 2), "%)", 
     "\nDownregulated: ", sum(final_result$sig == "downregulated"), 
     " (", round(mean(final_result$sig == "downregulated") * 100, 2), "%)"
    ), 
    size = 4, 
    hjust = 0, 
    color = "black"
  ) +
  # 添加 top 4 显著点的名字，使用 geom_text_repel 避免重叠
  geom_text_repel(
    data = top4_combined,
    aes(x = estimate, y = log_p, label = hormone),
    size = 2, 
    color = "black",
    box.padding = 0.5,  # 控制文本与点的距离
    point.padding = 0.5,  # 控制点与文本的距离
    max.overlaps = Inf  # 避免重叠
  )

dev.off()

png 
  2

In [7]:
save.image("/vol/projects/yzhang/500FG_aging/output/06_hormone/hormone_gender.Rdata")

In [1]:
load("/vol/projects/yzhang/500FG_aging/output/06_hormone/hormone_gender.Rdata")

In [3]:
final_result_sig <- final_result %>%
  filter(sig == "sig") %>%
  select(feature, estimate, p, padj)
head(final_result_sig)
dim(final_result_sig)
write_excel_csv(final_result_sig,"/vol/projects/yzhang/500FG_aging/output/06_hormone/hormone_age_liner_gender_FDR_sig.csv")

,feature,estimate,p,padj
,<chr>,<dbl>,<dbl>,<dbl>
Estimate,B_ADNC,-0.02501094,4.283096e-26,3.854787e-25
Estimate1,B_CORC,-0.02015736,2.037838e-17,9.170270e-17
Estimate2,B_DESC,-0.02048826,3.050586e-07,6.663984e-07
Estimate3,B_OHP,-0.01130764,3.791113e-03,4.874289e-03
Estimate4,B_PROG,-0.01576901,3.129267e-03,4.693901e-03
Estimate5,B_TESC,-0.01334730,2.096557e-09,6.289670e-09


[1] 8 4